In [ ]:
import geopandas as gpd
import folium
from folium import plugins
from IPython.display import display, clear_output

def start_interactive_mapping():
    # Direct URL for reliable loading
    url = "https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson"
    try:
        world = gpd.read_file(url)
        world.columns = [col.lower() for col in world.columns]
    except Exception as e:
        print(f"Error loading map data: {e}")
        return

    history = []

    print("--- Zoomable World Map (GitHub/Colab Version) ---")
    print("Type 'exit' to stop.")

    while True:
        # 1. User Input
        country_input = input("\nWhich country to add? ").strip().lower()
        if country_input == 'exit':
            print("Session ended.")
            break
            
        selected = world[world['name'].str.lower() == country_input]
        
        if selected.empty:
            print(f"Country '{country_input}' not found. Try 'Canada' or 'Brazil'.")
            continue

        marker_label = input(f"Enter label for {country_input.title()}: ")
        
        # 2. Get Coordinates
        point = selected.geometry.representative_point().iloc[0]
        history.append({
            'name': country_input.title(),
            'lat': point.y,
            'lon': point.x,
            'label': marker_label
        })

        # 3. Create Map with Corrected Attributions
        m = folium.Map(location=[point.y, point.x], zoom_start=3)

        # Built-in (No attribution string needed)
        folium.TileLayer('openstreetmap').add_to(m)

        # Custom Tiles (Attribution strings added to fix the ValueError)
        folium.TileLayer(
            tiles='https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png', 
            attr='Map data: &copy; OpenStreetMap contributors, SRTM | Map style: &copy; OpenTopoMap (CC-BY-SA)', 
            name="Topographic"
        ).add_to(m)

        # 4. Add Widgets
        plugins.Fullscreen().add_to(m)
        plugins.MousePosition().add_to(m)

        # 5. Add all markers from history
        for item in history:
            folium.Marker(
                location=[item['lat'], item['lon']],
                popup=f"<b>{item['name']}</b>: {item['label']}",
                tooltip=f"Click for info: {item['name']}",
                icon=folium.Icon(color='blue', icon='cloud')
            ).add_to(m)

        folium.LayerControl().add_to(m)

        # 6. Display in Cloud/Notebook
        clear_output(wait=True) # Clears the old map so only the updated one shows
        display(m)

if __name__ == "__main__":
    start_interactive_mapping()

--- Zoomable World Map (GitHub/Colab Version) ---
Type 'exit' to stop.
